# 📚 Section 10: Pandas + NumPy Integration
> Pandas + NumPy 실무 연동 — Pandas handles structure, NumPy handles computation, and real workflows constantly move data between them

---
# 🎯 Learning Objective
Today I want to learn:
- [x] How to convert between a Pandas `DataFrame` and a NumPy `ndarray` (`.to_numpy()` / `.values`) and back (`pd.DataFrame(arr, columns=...)`)
- [x] The extract-compute-reassign pattern: pull a column to `ndarray`, vectorize a KPI calculation, assign the result back as a new column
- [x] How to build a real end-to-end pipeline — data-quality check, cleaning, derived metrics, segmentation — combining Pandas structure with NumPy computation

---
# 🧠 Concept

## What is it?
*(Explain it in your own words.)*

In practice, Pandas and NumPy are never really separate — a Pandas `DataFrame` stores its data internally as NumPy arrays. Pandas handles the STRUCTURE (column names, row index, dtype bookkeeping), while NumPy is the CALCULATION ENGINE underneath. Real analysis work constantly shuttles data between the two: pull a column out as an `ndarray`, run a fast vectorized calculation, and assign the result back as a new column.

실무에서 Pandas와 NumPy는 사실 분리되어 있지 않습니다 — Pandas `DataFrame`은 내부적으로 데이터를 NumPy 배열로 저장합니다. Pandas는 구조(컬럼명, 행 인덱스, dtype 관리)를 담당하고, 그 아래에서 NumPy가 계산 엔진 역할을 합니다. 실제 분석 작업은 이 둘 사이를 끊임없이 오갑니다: 컬럼을 `ndarray`로 꺼내고, 빠른 벡터화 계산을 실행한 뒤, 결과를 새 컬럼으로 다시 대입합니다.

### Division of labor / 역할 분담

| Role / 역할 | Pandas | NumPy |
|---|---|---|
| Structure / 구조 | Column names, index, dtype (컬럼명, 인덱스, dtype) | — |
| Computation / 연산 | — | Vectorized math, aggregation (벡터화 연산, 집계) |
| Typical call / 대표 호출 | `df['col']`, `df.loc[]` | `arr.mean()`, `np.where()` |
| Conversion / 변환 | `df.to_numpy()` → ndarray | `pd.DataFrame(arr)` → DataFrame |

## Why do we use it?
*(When is it useful?)*

Because Pandas alone (via `.apply()`) is dozens of times slower than extracting a column to NumPy and vectorizing the same calculation — and because most real datasets need genuine data cleaning (missing values, outliers, wrong types) that this combination handles far more cleanly than either tool alone.

Pandas만으로(`.apply()`를 통해) 처리하면 컬럼을 NumPy로 꺼내 같은 계산을 벡터화하는 것보다 수십 배 느리기 때문이며, 실제 데이터셋 대부분이 결측치·이상치·잘못된 타입 같은 진짜 데이터 정제가 필요한데, 이 조합이 둘 중 하나만 쓸 때보다 훨씬 깔끔하게 처리하기 때문입니다.

## When is it used in Business Analytics?
*(Real-world use case)*

- Converting a DataFrame's numeric columns to an ndarray before feeding a model or running bulk math.  
  모델에 넣거나 대량 연산을 하기 전, DataFrame의 수치 컬럼을 ndarray로 변환할 때.
- Computing KPIs (margin, conversion rate, ROAS, MoM growth) as new DataFrame columns.  
  KPI(마진, 전환율, ROAS, 전월비 성장률)를 새 DataFrame 컬럼으로 계산할 때.
- Cleaning raw CSV-style data — type errors, missing values, negative outliers — before analysis.  
  분석 전 raw CSV 데이터(타입 오류, 결측치, 음수 이상치)를 정제할 때.
- Building customer segments (RFM) or channel/period performance reports end-to-end.  
  고객 세그먼트(RFM)나 채널·기간별 성과 리포트를 처음부터 끝까지 구축할 때.

---
# 📝 Syntax

## Basic Syntax

In [1]:
import numpy as np
import pandas as pd

df = pd.DataFrame({'revenue': [1200, 1350, 1100], 'cost': [800, 900, 750]})

# DataFrame -> ndarray
arr = df.to_numpy()               # official recommended way
print(arr, arr.dtype)

# Column -> ndarray -> vectorized calc -> new column
revenue = df['revenue'].to_numpy()
cost = df['cost'].to_numpy()
df['margin_pct'] = np.round((revenue - cost) / revenue * 100, 1)
print(df)

# ndarray -> DataFrame (column names required!)
result = pd.DataFrame(arr * 1.1, columns=['revenue_tax', 'cost_tax'])
print(result)

[[1200  800]
 [1350  900]
 [1100  750]] int64
   revenue  cost  margin_pct
0     1200   800        33.3
1     1350   900        33.3
2     1100   750        31.8
   revenue_tax  cost_tax
0       1320.0     880.0
1       1485.0     990.0
2       1210.0     825.0


## Common Variations

In [10]:
import pandas as pd

df = pd.DataFrame({'a': [1, 2, 3], 'b': [1.5, 2.5, 3.5]})

# mixed int + float columns -> to_numpy() upcasts EVERYTHING to float64
print(df.to_numpy().dtype)

# select specific columns before converting
sub = df[['a']].to_numpy()
print(sub.shape)   # (3, 1) -- stays 2D even for one column

# filter with Pandas, compute with NumPy
mask = df['a'] > 1
vals = df.loc[mask, 'b'].to_numpy()
print(vals.mean())

# IMPORTANT: to_numpy() can return a READ-ONLY array in recent pandas versions —
# always .copy() (or .astype()) before modifying it in place
safe_arr = df['b'].to_numpy().copy()
safe_arr[0] = 999
print(safe_arr)

float64
(3, 1)
3.0
[999.    2.5   3.5]


---
# 🧪 Small Examples

## Example 1: DataFrame ↔ ndarray — Sales Rep Performance Pipeline
### 10-1. DataFrame ↔ ndarray 변환

In [11]:
# Business: sales rep performance data
np.random.seed(42)
n = 50
df = pd.DataFrame({
    'rep_id': [f'REP{i:03d}' for i in range(1, n+1)],
    'region': np.random.choice(['Seoul', 'Gyeonggi', 'Busan', 'Daegu'], n),
    'revenue': np.random.randint(3000, 20000, n),
    'calls': np.random.randint(20, 150, n),
    'deals': np.random.randint(1, 15, n),
})

# Step 1: extract numeric columns to an ndarray
X = df[['revenue', 'calls', 'deals']].to_numpy()
print("Numeric matrix shape:", X.shape)

# Step 2: vectorized derived metrics
revenue, calls, deals = X[:, 0], X[:, 1], X[:, 2]
df['rpd'] = np.round(revenue / deals, 0)          # Revenue Per Deal
df['crm'] = np.round(deals / calls * 100, 2)       # Call-to-deal rate %

# Step 3: NumPy-based grade classification
p33, p67 = np.percentile(revenue, [33, 67])
df['grade'] = np.where(revenue >= p67, 'Gold', np.where(revenue >= p33, 'Silver', 'Bronze'))

# Step 4: region-level aggregation combining Pandas filtering + NumPy math
print("\nRegional summary:")
for region in df['region'].unique():
    rev_arr = df.loc[df['region'] == region, 'revenue'].to_numpy()
    print(f"  {region}: n={len(rev_arr)}, avg revenue={rev_arr.mean():,.0f}")

Numeric matrix shape: (50, 3)

Regional summary:
  Busan: n=13, avg revenue=12,107
  Daegu: n=16, avg revenue=11,832
  Seoul: n=10, avg revenue=9,243
  Gyeonggi: n=11, avg revenue=9,051


## Example 2: Column → Vector Op → New Column — Ad Campaign KPIs at Scale
### 10-2. Pandas 컬럼 → NumPy 벡터 연산 → 새 컬럼 추가

In [4]:
import numpy as np
import pandas as pd
import time

# Business: 10,000 ad campaigns -- compare .apply() vs vectorized NumPy
np.random.seed(7)
n = 10000
df = pd.DataFrame({
    'ad_spend': np.random.randint(50000, 2000000, n),
    'clicks': np.random.randint(10, 5000, n),
    'conversions': np.random.randint(0, 200, n),
    'revenue': np.random.randint(0, 5000000, n),
})

# Slow: Pandas .apply() row by row
start = time.time()
df['roas_slow'] = df.apply(
    lambda r: r['revenue'] / r['ad_spend'] if r['ad_spend'] > 0 else 0, axis=1)
slow_time = time.time() - start

# Fast: extract to ndarray, vectorize, assign back
start = time.time()
spend = df['ad_spend'].to_numpy()
clicks = df['clicks'].to_numpy()
conversions = df['conversions'].to_numpy()
revenue = df['revenue'].to_numpy()

df['roas'] = np.where(spend > 0, revenue / spend, 0)
df['cvr'] = np.where(clicks > 0, conversions / clicks * 100, 0)
df['cpa'] = np.where(conversions > 0, spend / conversions, np.nan)
fast_time = time.time() - start

print(f"Pandas apply: {slow_time:.3f}s")
print(f"NumPy vectorized: {fast_time:.4f}s")
print(f"Speedup: ~{slow_time/fast_time:.0f}x")

Pandas apply: 0.146s
NumPy vectorized: 0.0022s
Speedup: ~66x


/var/folders/9d/hwl9yp1n3rv1y9j90spl8rmh0000gn/T/ipykernel_2087/3535242565.py:30: RuntimeWarning: divide by zero encountered in divide
  df['cpa'] = np.where(conversions > 0, spend / conversions, np.nan)


## Example 3: A Messy-CSV Cleaning Pipeline
### 10-3. CSV 데이터 처리 실무 예시 — 전처리 파이프라인

In [5]:
import numpy as np
import pandas as pd

# Business: raw data with type errors, missing values, and negative outliers
raw = pd.DataFrame({
    'revenue': ['1200', '1350', '1100', 'ERR', '1600'],   # string + bad value
    'cost': [800, 900, None, 950, 1100],                   # missing value
    'visitors': [3200, 3500, 2800, -500, 4100],            # negative outlier
})

# Step 1: safe type conversion (bad strings -> NaN)
revenue_num = np.zeros(len(raw))
for i, v in enumerate(raw['revenue'].to_numpy()):
    try:
        revenue_num[i] = float(v)
    except (ValueError, TypeError):
        revenue_num[i] = np.nan
print("Converted revenue:", revenue_num)

# Step 2: fill missing values with the column mean (NaN-safe)
cost_arr = raw['cost'].to_numpy().astype(float)
cost_arr[np.isnan(cost_arr)] = np.nanmean(cost_arr)
print("Cost after fill:", cost_arr)

# Step 3: clip negative outliers
visitors_clean = np.clip(raw['visitors'].to_numpy().astype(float), 0, None)
print("Visitors after clip:", visitors_clean)

Converted revenue: [1200. 1350. 1100.   nan 1600.]
Cost after fill: [ 800.   900.   937.5  950.  1100. ]
Visitors after clip: [3200. 3500. 2800.    0. 4100.]


## Example 4: RFM Segmentation & Churn Risk Scoring
### 10-4. 고객 데이터 분석 실무 예시 — 세그먼트 & 이탈 탐지

In [6]:
import numpy as np
import pandas as pd

# Business: RFM segmentation for 200 customers
np.random.seed(42)
n = 200
df = pd.DataFrame({
    'customer_id': [f'C{i:04d}' for i in range(1, n+1)],
    'total_spend': np.random.randint(10000, 600000, n),
    'order_count': np.random.randint(1, 25, n),
    'days_since_buy': np.random.randint(1, 365, n),
})

spend = df['total_spend'].to_numpy()
orders = df['order_count'].to_numpy()
days = df['days_since_buy'].to_numpy()

# R/F/M scores (1-5, higher is better)
r_score = np.where(days <= 30, 5, np.where(days <= 90, 4,
          np.where(days <= 180, 3, np.where(days <= 270, 2, 1))))
f_score = np.digitize(orders, bins=[0, 3, 7, 12, 18])
m_score = np.digitize(spend, bins=[0, 50000, 150000, 300000, 450000])

df['RFM'] = r_score + f_score + m_score
df['segment'] = np.where(df['RFM'] >= 12, 'Champion',
                 np.where(df['RFM'] >= 9, 'Loyal',
                 np.where(df['RFM'] >= 6, 'Potential', 'At Risk')))

# Segment distribution
segs, counts = np.unique(df['segment'], return_counts=True)
for s, c in zip(segs, counts):
    print(f"{s}: {c} ({c/n*100:.1f}%)")

At Risk: 6 (3.0%)
Champion: 36 (18.0%)
Loyal: 100 (50.0%)
Potential: 58 (29.0%)


## Example 5: Channel × Month Revenue — Annual, Quarterly & MoM Reports
### 10-5. 매출 데이터 분석 실무 예시 — 채널별·기간별 집계

In [7]:
import numpy as np
import pandas as pd

# Business: 4 channels x 12 months revenue matrix
channels = ["Search", "SNS", "Email", "Direct"]
np.random.seed(1)
sales_matrix = np.array([
    np.random.randint(800, 1500, 12),
    np.random.randint(400, 900, 12),
    np.random.randint(200, 600, 12),
    np.random.randint(300, 700, 12),
])
months = [f"M{m}" for m in range(1, 13)]
df = pd.DataFrame(sales_matrix, index=channels, columns=months)

rev = df.to_numpy()   # (4, 12)

# Channel-level annual totals and contribution
annual = rev.sum(axis=1)
share = annual / annual.sum() * 100
print("Channel contribution:")
for ch, s in zip(channels, share):
    print(f"  {ch}: {s:.1f}%")

# Month-level totals, MoM growth, YTD
monthly_total = rev.sum(axis=0)
mom_growth = np.zeros(12)
mom_growth[1:] = (monthly_total[1:] - monthly_total[:-1]) / monthly_total[:-1] * 100
ytd = np.cumsum(monthly_total)

print("\nMonthly totals & MoM growth:")
for m, mt, gr in zip(months[:4], monthly_total[:4], mom_growth[:4]):
    print(f"  {m}: {mt:,} ({gr:+.1f}%)")
print(f"  ... YTD through M12: {ytd[-1]:,}")

Channel contribution:
  Search: 40.5%
  SNS: 26.3%
  Email: 14.9%
  Direct: 18.3%

Monthly totals & MoM growth:
  M1: 2,590 (+0.0%)
  M2: 2,849 (+10.0%)
  M3: 2,735 (-4.0%)
  M4: 3,163 (+15.6%)
  ... YTD through M12: 32,291


## Example 6: Practice — One Mini-Exercise per Subtopic  
### 연습 문제 (Practice Problems, 10-1 ~ 10-5)

**Task / 과제:** This section's original material has one practice problem per subtopic — solve all five below.  
이번 섹션은 하위주제마다 연습 문제가 하나씩 있습니다 — 아래 5개를 모두 풀어보세요.

1. **(10-1)** A products DataFrame (`price`, `sales`) → Min-Max normalize both numeric columns, add as `_normalized` suffix columns.  
   상품 DataFrame(`price`, `sales`) → 두 수치 컬럼을 Min-Max 정규화해서 `_normalized` 접미사 컬럼으로 추가하세요.
2. **(10-2)** A customer DataFrame (`total_spend`, `orders`, `visits`) → compute AOV (`spend/orders`) and `buy_rate` (`orders/visits*100`) as new columns.  
   고객 DataFrame(`total_spend`, `orders`, `visits`) → AOV(`spend/orders`)와 `buy_rate`(`orders/visits*100`)를 새 컬럼으로 계산하세요.
3. **(10-3)** A DataFrame with 2 numeric columns containing `NaN` and a negative value → fill `NaN` with the column mean, clip negatives to 0, print the remaining `NaN` count.  
   `NaN`과 음수값이 섞인 수치 컬럼 2개짜리 DataFrame → `NaN`은 열 평균으로, 음수는 0으로 clip한 뒤 남은 `NaN` 개수를 출력하세요.
4. **(10-4)** A customer DataFrame (`days_since_last`, `total_spend`) → flag at-risk customers (days > 90 AND spend < 50% of the mean), print count + percentage.  
   고객 DataFrame(`days_since_last`, `total_spend`) → 이탈 위험 고객(90일 초과 AND 평균의 50% 미만 지출)의 수와 비율을 출력하세요.
5. **(10-5)** A 3-product × 6-month sales array → print per-product half-year totals, per-month totals, and MoM growth.  
   3개 상품×6개월 판매량 배열 → 상품별 반기 합계, 월별 합계, 전월비 성장률을 출력하세요.

Fill in each `________` below, then run the cell.  
아래 각 `________`를 채운 후 셀을 실행하세요.

In [12]:
# ✏️ Practice — replace each ________ line below, then run this cell.
# ✏️ 연습 문제 — 아래 각 ________ 줄을 채운 후 셀을 실행하세요.

print("--- 10-1: DataFrame <-> ndarray ---")
# products df (price, sales) -> Min-Max normalize both cols as *_normalized new columns
products = pd.DataFrame({
    'product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor'],
    'price': [1200000, 25000, 45000, 350000],
    'sales': [120, 850, 430, 210],
})
X = products[['price', 'sales']].to_numpy().astype(float)
# axis=0 -> per-column min/max (a shared min/max across both columns would be meaningless)
# axis=0 -> 컬럼별 min/max (두 컬럼을 합친 min/max는 의미가 없음)
normalized = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
products[['price_normalized', 'sales_normalized']] = np.round(normalized, 3)
print(products)

print("\n--- 10-2: column -> vector op -> new column ---")
# customer df (total_spend, orders, visits) -> add aov and buy_rate columns
customers = pd.DataFrame({
    'customer_id': ['C001', 'C002', 'C003', 'C004'],
    'total_spend': [480000, 125000, 890000, 0],
    'orders': [6, 5, 12, 0],          # C004 has 0 orders -> divide-by-zero risk / 주문 0 -> 나눗셈 위험
    'visits': [40, 100, 60, 25],
})
spend = customers['total_spend'].to_numpy()
orders = customers['orders'].to_numpy()
visits = customers['visits'].to_numpy()

# Build a safe denominator FIRST -- np.where still evaluates both branches fully
# 분모를 먼저 안전하게 만들기 -- np.where는 양쪽 branch를 모두 계산함
safe_orders = np.where(orders > 0, orders, 1)
safe_visits = np.where(visits > 0, visits, 1)
customers['aov'] = np.where(orders > 0, spend / safe_orders, 0).round(0)
customers['buy_rate'] = np.where(visits > 0, orders / safe_visits * 100, 0).round(1)
print(customers)

print("\n--- 10-3: cleaning pipeline ---")
# df with 2 numeric cols (NaN + negative) -> fill NaN with mean, clip negatives to 0
raw = pd.DataFrame({
    'revenue': [1200.0, np.nan, 1100.0, -500.0, 1600.0],
    'visitors': [3200.0, 3500.0, np.nan, -800.0, 4100.0],
})
clean = raw.copy()
for col in ['revenue', 'visitors']:
    # .copy() -- to_numpy() can return a read-only array / 읽기 전용 배열이 반환될 수 있음
    arr = raw[col].to_numpy().astype(float).copy()
    arr[np.isnan(arr)] = np.nanmean(arr)      # nanmean, not mean -- NaN would poison it
    clean[col] = np.clip(arr, 0, None)        # None = no upper bound / 위쪽 한계 없음
print(clean)
print("remaining NaN:", clean.isna().sum().sum())

print("\n--- 10-4: RFM/churn ---")
# customer df (days_since_last, total_spend) -> flag at-risk (days>90 & spend<mean*0.5)
np.random.seed(42)
n = 200
cust = pd.DataFrame({
    'customer_id': [f'C{i:04d}' for i in range(1, n + 1)],
    'days_since_last': np.random.randint(1, 365, n),
    'total_spend': np.random.randint(10000, 600000, n),
})
days = cust['days_since_last'].to_numpy()
cust_spend = cust['total_spend'].to_numpy()

# Percentile/mean-based threshold adapts when the data shifts / 데이터가 바뀌면 기준도 따라감
threshold = cust_spend.mean() * 0.5
at_risk = (days > 90) & (cust_spend < threshold)   # & not 'and', each condition in parentheses
cust['at_risk'] = at_risk

print(f"spend mean={cust_spend.mean():,.0f}, threshold={threshold:,.0f}")
print(f"at-risk: {at_risk.sum()} / {n} ({at_risk.sum() / n * 100:.1f}%)")
print(cust[cust['at_risk']].head(3))

print("\n--- 10-5: channel/period aggregation ---")
# 3x6 product-sales array -> per-product totals, per-month totals, MoM growth
np.random.seed(1)
sales = np.random.randint(100, 900, (3, 6))
prod_names = ['P1', 'P2', 'P3']
months = [f'M{m}' for m in range(1, 7)]
print(pd.DataFrame(sales, index=prod_names, columns=months))

# The axis you pass is the one that disappears / axis에 넣은 축이 사라짐
prod_total = sales.sum(axis=1)      # (3,) -- per-product half-year total / 상품별 반기 합계
month_total = sales.sum(axis=0)     # (6,) -- per-month total / 월별 합계

# NumPy has no shift() -- slice one step apart instead / shift()가 없으니 슬라이싱으로 한 칸 밀기
mom = np.zeros(6)
mom[1:] = (month_total[1:] - month_total[:-1]) / month_total[:-1] * 100

print("\nper-product half-year totals:")
for name, t in zip(prod_names, prod_total):
    print(f"  {name}: {t:,}")
print("\nper-month totals & MoM growth:")
for m, t, g in zip(months, month_total, mom):
    print(f"  {m}: {t:,} ({g:+.1f}%)")

--- 10-1: DataFrame <-> ndarray ---
    product    price  sales  price_normalized  sales_normalized
0    Laptop  1200000    120             1.000             0.000
1     Mouse    25000    850             0.000             1.000
2  Keyboard    45000    430             0.017             0.425
3   Monitor   350000    210             0.277             0.123

--- 10-2: column -> vector op -> new column ---
  customer_id  total_spend  orders  visits      aov  buy_rate
0        C001       480000       6      40  80000.0      15.0
1        C002       125000       5     100  25000.0       5.0
2        C003       890000      12      60  74167.0      20.0
3        C004            0       0      25      0.0       0.0

--- 10-3: cleaning pipeline ---
   revenue  visitors
0   1200.0    3200.0
1    850.0    3500.0
2   1100.0    2500.0
3      0.0       0.0
4   1600.0    4100.0
remaining NaN: 0

--- 10-4: RFM/churn ---
spend mean=281,804, threshold=140,902
at-risk: 49 / 200 (24.5%)
   customer_id  days

---
# ⚠️ Common Mistakes

**Mistake 1 — Forgetting mixed dtypes upcast on `.to_numpy()`.**
If a DataFrame has both `int64` and `float64` columns, `.to_numpy()` upcasts EVERYTHING to `float64` — even the originally-clean integer column.  
DataFrame에 `int64`와 `float64` 컬럼이 섞여 있으면, `.to_numpy()`는 원래 깨끗한 정수였던 컬럼까지 포함해 전부 `float64`로 업캐스트합니다.  
✅ **Fix:** If you need to keep a column as integers, convert it back explicitly with `.astype(int)` after extracting.  
✅ **해결법:** 특정 컬럼을 정수로 유지해야 한다면, 꺼낸 뒤 `.astype(int)`로 명시적으로 다시 변환하세요.

**Mistake 2 — Skipping `columns=` when converting `ndarray` back to `DataFrame`.**
`pd.DataFrame(arr)` without `columns=` gives you `0, 1, 2...` as column names, losing any meaningful labels.  
`columns=` 없이 `pd.DataFrame(arr)`을 쓰면 컬럼명이 `0, 1, 2...`가 되어 의미 있는 라벨을 잃습니다.  
✅ **Fix:** Always pass `columns=[...]` explicitly when wrapping a NumPy result back into a DataFrame.  
✅ **해결법:** NumPy 결과를 DataFrame으로 다시 감쌀 때는 항상 `columns=[...]`을 명시적으로 전달하세요.

**Mistake 3 — Not guarding against divide-by-zero before assigning a ratio column.**
`df['roas'] = df['revenue'] / df['ad_spend']` becomes `inf` the instant any `ad_spend` is 0 — and that `inf` silently poisons any later `.mean()` on that column.  
`df['roas'] = df['revenue'] / df['ad_spend']`는 `ad_spend`가 0인 행이 하나라도 있으면 `inf`가 되며, 이 `inf`가 이후 그 컬럼의 `.mean()`을 조용히 오염시킵니다.  
✅ **Fix:** Always wrap risky divisions in `np.where(denominator > 0, ratio, fallback_value)`.  
✅ **해결법:** 위험한 나눗셈은 항상 `np.where(분모 > 0, 비율, 대체값)`으로 감싸세요.

**Mistake 4 — Not realizing `np.where(cond, a, b)` evaluates BOTH `a` and `b` fully.**
`np.where(conversions > 0, spend / conversions, np.nan)` still computes `spend / conversions` even where `conversions == 0`, producing a harmless-looking but noisy `RuntimeWarning: divide by zero`.  
`np.where(conversions > 0, spend / conversions, np.nan)`은 `conversions == 0`인 곳에서도 `spend / conversions`를 그대로 계산해서, 무해하지만 시끄러운 `RuntimeWarning: divide by zero`를 발생시킵니다.  
✅ **Fix:** The result is still correct (the warning-producing values get replaced), but if the warning noise bothers you, compute the safe version first: `safe_denom = np.where(conversions > 0, conversions, 1)`, then divide by that.  
✅ **해결법:** 결과 자체는 여전히 올바르지만(경고를 낸 값은 결국 교체됨), 경고가 거슬린다면 `safe_denom = np.where(conversions > 0, conversions, 1)`처럼 안전한 분모를 먼저 만들고 나누세요.

**Mistake 5 — Calling `.mean()` on a column that contains `NaN`.**
A plain `arr.mean()` returns `nan` if even one element is `NaN` — the missing value contaminates the whole calculation.  
일반 `arr.mean()`은 원소가 하나라도 `NaN`이면 `nan`을 반환합니다 — 결측값 하나가 전체 계산을 오염시킵니다.  
✅ **Fix:** Use `np.nanmean()` / `np.nanmedian()` when the array might contain `NaN`.  
✅ **해결법:** 배열에 `NaN`이 있을 수 있다면 `np.nanmean()` / `np.nanmedian()`을 사용하세요.

**Mistake 6 — Trying to modify a `.to_numpy()` result in place.**
In recent pandas versions, `df.to_numpy()` frequently returns a READ-ONLY array — assigning into it directly raises `ValueError: assignment destination is read-only`.  
최근 pandas 버전에서는 `df.to_numpy()`가 자주 읽기 전용(read-only) 배열을 반환합니다 — 직접 값을 대입하면 `ValueError: assignment destination is read-only`가 발생합니다.  
✅ **Fix:** Always call `.to_numpy().copy()` (or follow with `.astype(...)`, which also copies) before modifying the array in place.  
✅ **해결법:** 배열을 제자리에서 수정하기 전에는 항상 `.to_numpy().copy()`를 호출하거나 (마찬가지로 복사를 만드는) `.astype(...)`를 이어붙이세요.

**Mistake 7 — Forgetting `np.digitize()` bins start at index `0`, not `1`.**
`np.digitize(orders, bins=[0, 3, 7, 12, 18])` returns values `0` through `5` — if you want a familiar 1-5 style score, you need to add 1.  
`np.digitize(orders, bins=[0, 3, 7, 12, 18])`는 `0`부터 `5`까지의 값을 반환합니다 — 익숙한 1~5점 척도가 필요하다면 1을 더해야 합니다.  
✅ **Fix:** Check the actual output range with a quick `.min()`/`.max()` before trusting a `digitize()`-based score as-is.  
✅ **해결법:** `digitize()` 기반 점수를 그대로 신뢰하기 전에, `.min()`/`.max()`로 실제 출력 범위를 먼저 확인하세요.

**Mistake 8 — Confusing `axis=0` and `axis=1` on a channel × month matrix.**
This is the single most common mistake in period/channel reporting — `rev.sum(axis=1)` gives per-CHANNEL annual totals (one value per row), while `rev.sum(axis=0)` gives per-MONTH totals across all channels (one value per column).  
기간·채널 리포팅에서 가장 흔한 실수입니다 — `rev.sum(axis=1)`은 채널별 연간 합계(행마다 값 하나)를, `rev.sum(axis=0)`은 월별 전체 채널 합계(열마다 값 하나)를 줍니다.  
✅ **Fix:** Check the result's shape — `(n_channels,)` confirms `axis=1`; `(n_months,)` confirms `axis=0`.  
✅ **해결법:** 결과의 shape을 확인하세요 — `(채널 수,)`이면 `axis=1`, `(월 수,)`이면 `axis=0`이 맞습니다.

---
# 💡 Tips
Useful tips or shortcuts

- Whenever you're about to loop over DataFrame rows with `.apply(..., axis=1)`, stop and ask whether extracting the columns to NumPy arrays and vectorizing would work instead — it usually does, and it's dramatically faster.  
  DataFrame 행을 `.apply(..., axis=1)`로 반복하려 할 때는 잠깐 멈추고, 컬럼을 NumPy 배열로 꺼내 벡터화할 수는 없는지 생각해보세요 — 대부분 가능하고, 훨씬 빠릅니다.
- Boolean arrays multiplied by a weight and summed (`(condition1).astype(int)*3 + (condition2).astype(int)*2 + ...`) is a fast, readable way to build a multi-signal score like a churn-risk score.  
  불리언 배열에 가중치를 곱해 더하는 방식(`(조건1).astype(int)*3 + (조건2).astype(int)*2 + ...`)은 이탈 위험 점수 같은 다중 신호 점수를 빠르고 읽기 쉽게 만드는 방법입니다.
- `np.percentile()`-based thresholds (rather than hardcoded numbers) make a segmentation or scoring pipeline automatically adapt when the underlying data distribution shifts.  
  하드코딩된 숫자 대신 `np.percentile()` 기반 임계값을 쓰면, 데이터 분포가 바뀌어도 세그먼트·점수 파이프라인이 자동으로 적응합니다.

---
# 🔗 Related Concepts

```
Section 9 — Random / Sorting / Filtering
   (seed, rand/randn/randint/choice, sort/argsort, unique, clip, isnan/isinf)
        ↓
🔵 Section 10 — Pandas + NumPy Integration   ← you are here
   (to_numpy(), column -> vector op -> new column, real pipelines)
        ↓
Section 11 — Business KPI Calculations
   (growth rate, retention, CAC, LTV, ROAS, NPS...)
        ↓
Section 12 — Missing Value Handling
        ↓
Pandas → SQL → Tableau
```

*How is today's topic connected to other concepts?*

This section is where the previous 9 sections' individual tools (creation, indexing, aggregation, sorting, random generation) come together into real pipelines — and it's also the direct bridge into the dedicated Pandas guide. Every business KPI in Section 11 (growth rate, LTV, ROAS...) is built with exactly the extract-compute-reassign pattern practiced here.

이번 섹션은 앞선 9개 섹션의 개별 도구(생성, 인덱싱, 집계, 정렬, 난수 생성)가 실제 파이프라인으로 합쳐지는 지점이며, 별도의 Pandas 가이드로 곧장 이어지는 다리이기도 합니다. 섹션 11의 모든 비즈니스 KPI(성장률, LTV, ROAS...)는 정확히 여기서 연습한 "추출-계산-재대입" 패턴으로 만들어집니다.

---
# 💼 Business Example
*How would a Business Analyst use this?*

**Scenario / 시나리오:**
You need to build a lightweight weekly channel-performance snapshot: pull this week's spend/revenue into `ndarray`, compute ROAS and revenue share safely (one channel has zero spend), and flag any channel performing below a 2.0x ROAS target.
가벼운 주간 채널 성과 스냅샷을 만들어야 하는 상황입니다: 이번 주 지출·매출을 `ndarray`로 꺼내고, ROAS와 매출 비중을 안전하게 계산하며(한 채널은 지출이 0), ROAS 목표 2.0x 미만인 채널을 표시합니다.

**To-do / 할 일:**
- [x] Extract spend and revenue columns to `ndarray`.  
      지출과 매출 컬럼을 `ndarray`로 추출합니다.
- [x] Compute ROAS with a divide-by-zero guard.  
      0 나눗셈을 방어하며 ROAS를 계산합니다.
- [x] Compute each channel's share of total revenue.  
      각 채널의 전체 매출 대비 비중을 계산합니다.
- [x] Flag channels with ROAS below 2.0 (excluding the zero-spend channel).  
      ROAS가 2.0 미만인 채널을 표시합니다(지출 0인 채널은 제외).

In [13]:
import numpy as np
import pandas as pd

# Business: weekly channel performance snapshot, with a data-quality wrinkle baked in
df = pd.DataFrame({
    'channel': ['Search', 'SNS', 'Email', 'Direct'],
    'spend': [420000, 310000, 90000, 0],       # Direct has 0 spend (organic)
    'revenue': [980000, 540000, 210000, 150000],
})

spend = df['spend'].to_numpy()
revenue = df['revenue'].to_numpy()

# KPI 1: ROAS, guarded against divide-by-zero
df['roas'] = np.where(spend > 0, revenue / spend, np.nan)

# KPI 2: revenue share
df['share_pct'] = np.round(revenue / revenue.sum() * 100, 1)

# Flag underperformers: ROAS below 2.0 (excluding the organic/no-spend channel)
df['flag'] = np.where((df['roas'].to_numpy() < 2.0) & (spend > 0), 'below target', 'OK')

print(df)

  channel   spend  revenue      roas  share_pct          flag
0  Search  420000   980000  2.333333       52.1            OK
1     SNS  310000   540000  1.741935       28.7  below target
2   Email   90000   210000  2.333333       11.2            OK
3  Direct       0   150000       NaN        8.0            OK


/var/folders/9d/hwl9yp1n3rv1y9j90spl8rmh0000gn/T/ipykernel_2087/2156744540.py:15: RuntimeWarning: divide by zero encountered in divide
  df['roas'] = np.where(spend > 0, revenue / spend, np.nan)


---
# 📝 Summary
*Write today's concept in 3~5 sentences.*

Real analysis work moves constantly between Pandas and NumPy: `.to_numpy()`/`.values` pulls a `DataFrame` (or one column) out as an `ndarray`, NumPy runs the fast vectorized math, and the result is assigned straight back as a new column — dramatically faster than `.apply()` and the backbone of every KPI calculation, data-cleaning pipeline, customer segmentation, and channel/period report in real BA work. The recurring gotchas are all about staying alert at the boundary between the two tools: mixed dtypes upcast on conversion, missing `columns=` loses your labels, unguarded division creates `inf`, and `NaN` silently poisons a plain `.mean()` unless you reach for `np.nanmean()`.

실제 분석 작업은 Pandas와 NumPy 사이를 끊임없이 오갑니다: `.to_numpy()`/`.values`로 `DataFrame`(또는 컬럼 하나)을 `ndarray`로 꺼내고, NumPy가 빠른 벡터화 연산을 수행하고, 결과를 새 컬럼으로 바로 대입합니다 — `.apply()`보다 훨씬 빠르며, 실무의 모든 KPI 계산·데이터 정제 파이프라인·고객 세그먼트·채널/기간 리포트의 기반입니다. 반복적으로 나타나는 함정은 모두 두 도구의 경계에서 주의를 기울여야 하는 지점들입니다: 혼합 dtype은 변환 시 업캐스트되고, `columns=`를 생략하면 라벨을 잃고, 방어 코드 없는 나눗셈은 `inf`를 만들고, `NaN`은 `np.nanmean()`을 쓰지 않으면 일반 `.mean()`을 조용히 오염시킵니다.

---
# 📌 One Sentence Summary
Today's topic in ONE sentence.

> Pandas handles structure and NumPy handles computation, and the extract-compute-reassign pattern (`.to_numpy()` → vectorized math → `df['new_col'] = result`) is the backbone of every real KPI, cleaning, and segmentation pipeline.

> Pandas는 구조를, NumPy는 연산을 담당하며, "추출-계산-재대입" 패턴(`.to_numpy()` → 벡터화 연산 → `df['new_col'] = 결과`)이 실무의 모든 KPI·정제·세그먼트 파이프라인의 기반입니다.

---
# ❓ Review Questions

**Q1.** What happens to the dtype of `df.to_numpy()` if the DataFrame has both `int64` and `float64` columns?
DataFrame에 `int64`와 `float64` 컬럼이 섞여 있으면 `df.to_numpy()`의 dtype은 어떻게 되나요?

**Q2.** Why is `df['new_col'] = numpy_array` so much faster than `df.apply(lambda row: ..., axis=1)`?
`df['new_col'] = numpy_array`가 `df.apply(lambda row: ..., axis=1)`보다 왜 훨씬 빠른가요?

**Q3.** Why does `revenue / ad_spend` need a `np.where(ad_spend > 0, ...)` guard in real data?
실제 데이터에서 `revenue / ad_spend`에 왜 `np.where(ad_spend > 0, ...)` 방어 코드가 필요한가요?

**Q4.** Why does `arr.mean()` return `nan` if `arr` contains even one `NaN`, and what should you use instead?
`arr`에 `NaN`이 하나라도 있으면 `arr.mean()`이 왜 `nan`을 반환하며, 대신 무엇을 사용해야 하나요?

**Q5.** In a channel × month revenue matrix, which `axis` gives you the annual total per channel, and which gives you the total per month across all channels?
채널×월 매출 행렬에서, 채널별 연간 합계를 구하려면 어떤 `axis`를, 월별 전체 채널 합계를 구하려면 어떤 `axis`를 사용해야 하나요?

---
*📅 Try answering these again in a few days.*